In [0]:
-- ===========================================================================================================================
-- PARIS URBAN MOBILITY & INFRASTRUCTURE - GEOSPATIAL LAYER
-- POSTGIS SQL ANALYTICS 
-- ===========================================================================================================================
-- Purpose :
-- Use PostGIS spatial functions to perform urban analytics on Parisian infrastructure, examining the relationships
-- between city districts, bike-share mobility networks, road closures, and construction worksites.
-- Dataset grain :
--   - arrondissements: One record represents a municipal district in Paris with its boundary polygon.
--   - quartier_paris: One record represents a specific neighborhood with its boundary polygon and surface area.
--   - velib_emplacement_des_stations: One record represents a specific bike-sharing station point and its bike capacity.
--   - chantiers_perturbants: One record represents an active disruptive construction worksite with its spatial footprint.
--   - fermetures_voirie: One record represents a designated road closure line segment with its physical state and direction.
-- Main columns :
--   - arrondissements: c_ar, l_ar, geom
--   - quartier_paris: l_qu, surface, geom
--   - velib_emplacement_des_stations: stationcode, name, capacity, geom
--   - chantiers_perturbants: identifiant, maitre_ouvrage, voie, geom
--   - fermetures_voirie: type_fv, sens_fv, etat_avancement, geom
-- Assumed schema and table names :
-- workspace.geojson_files.arrondissements, workspace.geojson_files.quartier_paris, workspace.geojson_files.velib_emplacement_des_stations,
-- workspace.geojson_files.chantiers_perturbants, workspace.geojson_files.fermetures_voirie
-- Analytics objective :
-- Apply spatial joins (ST_Intersects, ST_Contains), proximity mapping (ST_DWithin, ST_Distance), and geometric
-- transformations (ST_Transform to SRID 2154) to calculate true meter-based area, perimeter, and distance metrics,
-- ultimately analyzing the spatial impact of urban worksites on mobility infrastructure.
-- ===========================================================================================================================


-- Display arrondissements table 

SELECT * 
FROM workspace.geojson_files.arrondissements ;

-- Display chantiers table 

SELECT * 
FROM workspace.geojson_files.chantiers_perturbants ; 

-- Display fermetures voiries table 

SELECt * 
FROM workspace.geojson_files.fermetures_voirie ; 

-- Display quartiers table 

SELECT * 
FROM workspace.geojson_files.quartier_paris ; 

-- Display velib 

SELECT * 
FROM workspace.geojson_files.velib_emplacement_des_stations ;

-- neighborhood count 

SELECT  COUNT(*) AS neighborhood_count 
FROM workspace.geojson_files.quartier_paris ; 

-- Vélib’ station records in the dataset

SELECT COUNT(*) AS stations_count
FROM workspace.geojson_files.velib_emplacement_des_stations ;

-- geometry type for each neighborhood

SELECT l_qu , ST_GeometryType (geom) AS geometry_type  
FROM workspace.geojson_files.quartier_paris ; 

--  perimeter of each neighborhood polygon. 

SELECT l_qu , ST_Perimeter(ST_Transform(geom,2154)) AS perimeter 
FROM workspace.geojson_files.quartier_paris 
ORDER BY perimeter DESC ; 

--  area of each neighborhood polygon 

SELECT l_qu,ST_Area(geom) AS area 
FROM workspace.geojson_files.quartier_paris 
ORDER BY area DESC ; 

--  area of each neighborhood polygon in m2

SELECT l_qu,ST_Area(ST_Transform(geom,2154)) AS area 
FROM workspace.geojson_files.quartier_paris 
ORDER BY area DESC ; 

-- comparaison between the provided surface with the geometry-derived area

SELECT l_qu,
    ROUND(surface,2),
    ROUND(ST_Area(ST_Transform(geom,2154)),2),
    CASE WHEN  ROUND(surface,2) == (ROUND(ST_Area(ST_Transform(geom,2154)),2)) THEN TRUE
    WHEN ROUND(surface,2) != (ROUND(ST_Area(ST_Transform(geom,2154)),2)) THEN FALSE END AS equal 
FROM workspace.geojson_files.quartier_paris ;

-- centroid of each neighborhood. 

SELECT l_qu , ST_AsText(ST_Centroid(geom)) AS centroid 
FROM workspace.geojson_files.quartier_paris ; 

-- Vélib’ stations count inside each neighborhood polygon 

SELECT q.l_qu , COUNT (stationcode) AS stations_count
FROM workspace.geojson_files.velib_emplacement_des_stations s  
JOIN workspace.geojson_files.quartier_paris q on st_within(s.geom ,q.geom) 
GROUP BY q.l_qu 
ORDER BY stations_count DESC ; 

--   List the station names and neighborhood names for stations inside a neighborhood.

SELECT q.l_qu,s.name AS station_name , s.stationcode 
FROM workspace.geojson_files.quartier_paris q 
LEFT JOIN workspace.geojson_files.velib_emplacement_des_stations s ON ST_Contains(q.geom,s.geom) 
ORDER BY q.l_qu;

-- station points that intersect a particular neighborhood

SELECT * 
FROM workspace.geojson_files.velib_emplacement_des_stations s
JOIN workspace.geojson_files.quartier_paris q ON ST_Intersects(s.geom,q.geom) 
WHERE q.l_qu = 'Odéon';

-- stations count in each arrondissement 

SELECT c_ar,COUNT ( stationcode )  AS stations_count
FROM workspace.geojson_files.arrondissements a 
JOIN workspace.geojson_files.velib_emplacement_des_stations s ON ST_Contains(a.geom,s.geom)
GROUP BY c_ar 
ORDER BY stations_count DESC;

-- Chantiers geometry type 

SELECT identifiant, ST_GeometryType(geom)
FROM workspace.geojson_files.chantiers_perturbants ; 

-- worksite in neighberhoods 

SELECT l_qu,COUNT ( identifiant ) AS chantiers_count
FROM workspace.geojson_files.chantiers_perturbants c 
JOIN workspace.geojson_files.quartier_paris q ON ST_within(c.geom,q.geom)
GROUP BY l_qu 
ORDER BY chantiers_count DESC;

-- stations that are outside neighborhoods

SELECT * 
FROM workspace.geojson_files.velib_emplacement_des_stations s
LEFT ANTI JOIN workspace.geojson_files.quartier_paris q  ON ST_WithIn(s.geom,q.geom) ; 

--  available capacity bikes per neighborhood 

SELECT l_qu AS neighborhood , SUM (capacity)  AS total 
FROM workspace.geojson_files.velib_emplacement_des_stations s 
JOIN workspace.geojson_files.quartier_paris q on ST_Contains(q.geom,s.geom) 
GROUP BY neighborhood 
ORDER BY total DESC;

--  100-metre buffer around every station

SELECT stationcode, ST_AsText(ST_Buffer(geom,100)) AS buffer
FROM workspace.geojson_files.velib_emplacement_des_stations ; 

-- stations within 500 metres of station 13103

WITH stationsdistance AS (
SELECT 
    s.stationcode,
    s.name,
    ST_Distance(
            ST_Transform(s.geom,2154),
            ST_Transform(target.geom,2154)) AS distance
FROM workspace.geojson_files.velib_emplacement_des_stations s 
 CROSS JOIN 
 (
    SELECT  geom 
    FROM workspace.geojson_files.velib_emplacement_des_stations
    WHERE stationcode = 13103
 ) AS target 
 )
 SELECT * 
 FROM stationsdistance 
 WHERE distance <= 500 AND stationcode != 13103
; 

-- stations within a 500-metre buffer of each neighborhood centroid.

WITH centroids AS 
(
SELECT l_qu , ST_Transform(ST_Centroid(geom),2154) AS centroid 
FROM workspace.geojson_files.quartier_paris 
)
, buffers AS 
(
SELECT stationcode , ST_Buffer(ST_Transform(geom,2154),500) AS buffer 
FROM workspace.geojson_files.velib_emplacement_des_stations )
SELECT l_qu,stationcode 
FROM centroids c 
JOIN buffers b ON ST_Within (c.centroid,b.buffer) 
ORDER BY l_qu; 

-- stations count  within a 300-metre buffer of each neighborhood centroid

WITH centers AS (
  SELECT l_qu, ST_Centroid(st_transform(geom,2154)) AS center_geom
  FROM workspace.geojson_files.quartier_paris
)
SELECT c.l_qu, COUNT(s.stationcode) AS stations_near_center
FROM centers c
LEFT JOIN workspace.geojson_files.velib_emplacement_des_stations s
  ON ST_Distance(c.center_geom, ST_transform(s.geom,2154)) <= 500
GROUP BY c.l_qu
ORDER BY stations_near_center DESC;

-- Measure Road Closure Lengths in Kilometers by Infrastructure Type 

SELECT  
  type_fv ,
  ROUND (SUM (st_length( st_transform(geom ,2154 ))/1000),2) AS length_km
FROM workspace.geojson_files.fermetures_voirie 
GROUP BY type_fv; 

-- Top 5 Largest Disruptive Construction Worksites by Footprint Area

SELECT identifiant,maitre_ouvrage,ROUND(ST_Area(st_transform(geom,2154)),2) AS area_m2
FROM workspace.geojson_files.chantiers_perturbants 
ORDER BY area_m2 DESC 
LIMIT 5 ;

--  Road Closures Crossing Multiple Arrondissements 

SELECT c.identifiant, COUNT(DISTINCT a.c_ar) AS arrondissements_crossed 
FROM workspace.geojson_files.chantiers_perturbants c 
JOIN workspace.geojson_files.arrondissements a ON ST_Intersects(c.geom, a.geom)
GROUP BY c.identifiant
HAVING COUNT(DISTINCT a.c_ar) > 1;

--  Neighboring Arrondissements Sharing a Border with the 1st Arrondissemen

SELECT a1.l_ar, a1.c_ar, a2.c_ar  
FROM workspace.geojson_files.arrondissements a1
JOIN workspace.geojson_files.arrondissements a2 ON ST_touches(a1.geom,a2.geom)
  WHERE a1.c_ar=1 AND a2.c_ar != 1 ; 

-- Road Closure Lengths by Status  and Direction

SELECT 
  sens_fv,
  etat_avancement, 
  ROUND(SUM (st_length(st_transform(geom,2154))),2) AS total_length
FROM workspace.geojson_files.fermetures_voirie 
WHERE sens_fv IS NOT NULL 
GROUP BY sens_fv,etat_avancement
ORDER BY sens_fv;

-- Vélib' Stations Within 150 Meters of an Active Worksite 


WITH buffers  AS 
(
SELECT 
  stationcode,
  name,
  ST_buffer(ST_transform(geom,2154),150) AS buffer 
FROM workspace.geojson_files.velib_emplacement_des_stations 
)
SELECT stationcode,name
FROM  buffers 
JOIN workspace.geojson_files.chantiers_perturbants c  ON ST_intersects (ST_transform(c.geom,2154),buffer)

-- 3 Nearest Vélib' Stations to the Place de la Nation Worksite

WITH stations AS 
(
SELECT 
  s.stationcode, 
  s.name , 
  s.geom ,
  ST_Distance (ST_transform(s.geom,2154),
    ST_transform(target.geom,2154)) AS distance
FROM workspace.geojson_files.velib_emplacement_des_stations s 
CROSS JOIN 
(SELECT ST_centroid(geom) AS geom
FROM workspace.geojson_files.chantiers_perturbants 
WHERE voie = 'Place de la Nation') AS target
 )
SELECT 
  stationcode,
  name ,
  distance
FROM stations 
ORDER BY distance 
LIMIT 3   ; 



